# B2_tinyvec_service: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/bonus/B2_tinyvec_service/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Inspect write-ahead ordering and incomplete-tail recovery.
Concepts: durability and replay.
Run: uv run python modules/bonus/B2_tinyvec_service/examples/01_from_scratch.py
Expected: assertions pass, then measured values or a clearly labeled illustrative result.
CPU: seconds to a minute; frozen MiniLM or explicit public subset, no model downloads.
Native profiles and optional packages are described in README.md.
"""

import tempfile
from pathlib import Path

import numpy as np

from common.stores.types import CollectionSpec, Record
from tinyvec.durable import DurableTinyVec


def main() -> None:
    with tempfile.TemporaryDirectory() as temporary:
        root = Path(temporary)
        spec = CollectionSpec("lesson", 2)
        with DurableTinyVec(root, spec) as store:
            print("acknowledged", store.upsert([Record("a", np.array([1.0, 0.0]))]))
            store.checkpoint()
        with (root / "mutations.jsonl").open("ab") as stream:
            stream.write(b"{incomplete")
        with DurableTinyVec(root, spec) as restored:
            assert restored.search([1, 0])[0].id == "a"
            print("recovered sequence", restored.wal.sequence)


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/bonus/B2_tinyvec_service/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Compare an acknowledged write with native SQLite transaction recovery.
Concepts: durability and replay.
Run: uv run python modules/bonus/B2_tinyvec_service/examples/02_production.py
Expected: assertions pass, then measured values or a clearly labeled illustrative result.
CPU: seconds to a minute; frozen MiniLM or explicit public subset, no model downloads.
Native profiles and optional packages are described in README.md.
"""

import sqlite3
import subprocess
import sys
import tempfile
from contextlib import closing
from pathlib import Path


def main() -> None:
    with tempfile.TemporaryDirectory() as temporary:
        path = Path(temporary) / "native.db"
        code = "import sqlite3,sys,os; d=sqlite3.connect(sys.argv[1]); d.execute('PRAGMA journal_mode=WAL'); d.execute('PRAGMA synchronous=FULL'); d.execute('CREATE TABLE records(id TEXT PRIMARY KEY,value TEXT)'); d.execute(\"INSERT INTO records VALUES ('a','committed')\"); d.commit(); os._exit(23)"
        child = subprocess.run([sys.executable, "-c", code, str(path)], check=False)
        assert child.returncode == 23
        with closing(sqlite3.connect(path)) as db:
            assert db.execute("SELECT value FROM records").fetchone()[0] == "committed"
        print("SQLite committed row survived abrupt process exit; no power-loss claim")


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 03_measure

Source: `modules/bonus/B2_tinyvec_service/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("fastapi") is None:
    print("SKIP native comparison: --extra apps")
else:
    """Send concurrent operations through the durable HTTP service and restart it.
    Concepts: durability and replay.
    Run: uv run python modules/bonus/B2_tinyvec_service/examples/03_measure.py
    Expected: assertions pass, then measured values or a clearly labeled illustrative result.
    CPU: seconds to a minute; frozen MiniLM or explicit public subset, no model downloads.
    Native profiles and optional packages are described in README.md.
    """
    # course-extra: apps

    import tempfile
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path

    from fastapi.testclient import TestClient

    from common.research import record
    from tinyvec.service import create_app

    def main() -> None:
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            with TestClient(create_app(root)) as client:

                def send(i):
                    response = client.post(
                        "/records", json={"records": [{"id": str(i), "vector": [1.0, i / 100]}]}
                    )
                    assert response.status_code == 200
                    return response.json()["sequence"]

                with ThreadPoolExecutor(max_workers=4) as pool:
                    sequences = list(pool.map(send, range(20)))
                assert sorted(sequences) == list(range(1, 21))
            with TestClient(create_app(root)) as client:
                recovered = client.get("/health").json()["sequence"]
                assert recovered == 20
            print(
                record(
                    "durable-service",
                    {
                        "concurrent_clients": 4,
                        "acknowledged": 20,
                        "recovered_sequence": recovered,
                        "scope": "Loopback test transport; serialized storage lock, no power-cut simulation",
                    },
                    "python modules/bonus/B2_tinyvec_service/examples/03_measure.py",
                )
            )

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
import tempfile
from pathlib import Path

from tinyvec.wal import WriteAheadLog

with tempfile.TemporaryDirectory() as d:
    log = WriteAheadLog(Path(d) / "log.jsonl")
    log.append("delete", ["withdrawn"])
    print(log.replay())

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.